# Laya-1.2-speedup — Full Standalone **FP16** Export (Colab)

This is the **FP16-checkpoint version** of the [original standalone export notebook](https://github.com/vtavakkoli/TinyCeNN-LM/blob/3996018885992df1ac5c5d1803b91aacd33a2cf7/notebooks/Laya_1p2_Speedup_Full_Standalone_Export_Colab.ipynb), pinned to commit `3996018885992df1ac5c5d1803b91aacd33a2cf7`.

- Exports all **28 ModernBERT layers**, the V4.6 optimized **64-token decision head**, type embeddings, scorer, tokenizer, and architecture code. **No training or FP32 checkpoint is needed at inference.** The unused `act_head` is not exported, as in the source notebook.
- **All floating-point weights and floating buffers are physically serialized as FP16** in a single `model.safetensors` (not merely FP16 autocast on FP32 weights). Integer and boolean state retains its original dtype.
- The exported `modeling_laya_standalone.py` **loads the FP16 checkpoint as an actual FP16 model** from local files, including offline reload and a `predict` convenience API.
- Includes checkpoint dtype inspection, reload/decision-probability parity validation, full disk-size report, source provenance, model card, and optional Hugging Face publishing to **`vtava/Laya-1.2-speedup-standalone-fp16`**, private by default. Existing models are not overwritten by default.

**Requirements:** In Colab select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then **Runtime → Run all**. Downloads from the source repos and GitHub happen **only when generating** the export. The resulting model directory has **no TinyCeNN/GitHub/upstream dependency at inference**. FP16 may not reproduce all FP32 decisions; reported parity is specifically *pre-save FP16 vs local-reloaded FP16*. This notebook does not claim a newly measured 1.2× speedup or held-out accuracy.


In [ ]:
#@title 1. Install pinned build dependencies and check GPU
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@8ba9b41894c5f0708729ad8720c2b5b3d4eafa20"
import ast, gc, importlib.util, json, os, sys, urllib.request
from collections import OrderedDict
from pathlib import Path
import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, notebook_login
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    build_checkpoint_model, normalize_question, build_sequence, collate_items, QTYPES)
assert torch.cuda.is_available(), "Select a CUDA GPU runtime in Colab (Runtime > Change runtime type)."
DEVICE = "cuda"
FP16 = torch.float16
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())


In [ ]:
#@title 2. Configure destination and load source model (export-time only)
HEAD_REPO = "vtava/Laya-1.2-speedup"
DEST_REPO = "vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
PUBLISH_TO_HF = True #@param {type:"boolean"}
PRIVATE_REPO = True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING = False #@param {type:"boolean"}
OUT = Path("/content/Laya_1p2_FP16_Standalone")
DEST = OUT / "standalone_model"
(DEST / "encoder").mkdir(parents=True, exist_ok=True)
(DEST / "tokenizer").mkdir(parents=True, exist_ok=True)
assert DEST_REPO.startswith("vtava/") and DEST_REPO != HEAD_REPO, "Select a NEW vtava model repository."
api = HfApi()
try:
    published_info = api.model_info(HEAD_REPO)
except Exception as exc:
    print("Source head may require authentication:", repr(exc))
    notebook_login()
    published_info = api.model_info(HEAD_REPO)
head_cfg = json.loads(Path(hf_hub_download(HEAD_REPO, "config.json")).read_text())
assert head_cfg["source_model"] == "convaiinnovations/laya-typed-decisions", head_cfg
assert int(head_cfg["head_context_budget"]) == 64, head_cfg
assert head_cfg.get("all_gates_passed") is True, "Source head does not report passed gates."
SOURCE = head_cfg["source_model"]
SOURCE_REVISION = head_cfg.get("source_revision") or api.model_info(SOURCE).sha
SOURCE_DIR = Path(snapshot_download(SOURCE, revision=SOURCE_REVISION, allow_patterns=[
    "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"
]))
HEAD_FILE = Path(hf_hub_download(HEAD_REPO, "head.safetensors"))
PUBLISHED_CODE = Path(hf_hub_download(HEAD_REPO, "modeling_laya_speedup.py"))
original, base_config = build_checkpoint_model(SOURCE_DIR)
original.head.load_state_dict(load_file(str(HEAD_FILE), device="cpu"), strict=True)
original = original.to(DEVICE).eval().requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained(str(SOURCE_DIR / "tokenizer"))
assert tokenizer.pad_token_id is not None
print("Source:", SOURCE, SOURCE_REVISION)
print("Optimized head:", HEAD_REPO, published_info.sha)
print("Base checkpoint bytes:", (SOURCE_DIR / "model.safetensors").stat().st_size)
print("Head checkpoint bytes:", HEAD_FILE.stat().st_size)


In [ ]:
#@title 3. Embed the original standalone runtime and patch the local loader for native FP16
# Extract the runtime *source string* from the exact original notebook at export time.
# It is written INTO the exported folder. Local inference never calls GitHub.
PINNED_COMMIT = "3996018885992df1ac5c5d1803b91aacd33a2cf7"
UPSTREAM_NOTEBOOK = (
    f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{PINNED_COMMIT}/"
    "notebooks/Laya_1p2_Speedup_Full_Standalone_Export_Colab.ipynb"
)
with urllib.request.urlopen(UPSTREAM_NOTEBOOK, timeout=90) as response:
    upstream = json.loads(response.read().decode("utf-8"))
runtime_cells = ["".join(cell.get("source", [])) for cell in upstream["cells"]
                 if "STANDALONE_CODE = " in "".join(cell.get("source", []))]
assert len(runtime_cells) == 1, "Pinned upstream standalone runtime source was not found."
tree = ast.parse(runtime_cells[0])
assignments = [stmt for stmt in tree.body if isinstance(stmt, ast.Assign)
               and any(isinstance(t, ast.Name) and t.id == "STANDALONE_CODE" for t in stmt.targets)]
assert len(assignments) == 1
STANDALONE_CODE = ast.literal_eval(assignments[0].value)
# Force half parameters and buffers before loading the saved half checkpoint.
old = """    model.load_state_dict(state,strict=True)
    model=model.to(device).eval().requires_grad_(False)"""
new = """    model=model.to(dtype=torch.float16)
    model.load_state_dict(state,strict=True)
    model=model.to(device).eval().requires_grad_(False)"""
assert STANDALONE_CODE.count(old) == 1, "Runtime loader changed upstream; manual review required."
STANDALONE_CODE = STANDALONE_CODE.replace(old, new)
# Reject accidentally using the FP32 or unrelated standalone repository at inference.
old_meta = '    assert metadata["format"]=="laya-1p2-full-standalone-v1"'
new_meta = old_meta + '\n    assert metadata.get("dtype_on_disk")=="float16", "Expected FP16 standalone checkpoint"'
assert STANDALONE_CODE.count(old_meta) == 1
STANDALONE_CODE = STANDALONE_CODE.replace(old_meta, new_meta)
code_path = DEST / "modeling_laya_standalone.py"
code_path.write_text(STANDALONE_CODE, encoding="utf-8")
compile(STANDALONE_CODE, str(code_path), "exec")
spec = importlib.util.spec_from_file_location("laya_fp16_standalone", str(code_path))
assert spec and spec.loader
standalone_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(standalone_mod)
assert hasattr(standalone_mod, "LayaStandalone") and hasattr(standalone_mod, "load_model")
print("Wrote self-contained FP16 inference runtime:", code_path, code_path.stat().st_size, "bytes")


In [ ]:
#@title 4. Check architecture parity BEFORE casting; build audit inputs
pspec = importlib.util.spec_from_file_location("published_laya_speedup", str(PUBLISHED_CODE))
assert pspec and pspec.loader
published_mod = importlib.util.module_from_spec(pspec)
pspec.loader.exec_module(published_mod)
published_model = published_mod.MarkerAwareScores(original, budget=64).eval()
full_model = standalone_mod.LayaStandalone(budget=64, pretrained_components=original).eval()
Q = {
    "steering": {"type": "choice", "instructions": "Choose road direction.",
                 "criteria": {"left": "road curves left", "straight": "road is straight", "right": "road curves right"}},
    "speed": {"type": "choice", "instructions": "Choose safe longitudinal intent.",
              "criteria": {"accelerate": "gain speed", "hold": "maintain speed", "brake": "reduce speed"}},
}
STATES = [
    {"road_direction": "left", "road_visibility": 0.98, "speed": 7.0},
    {"road_direction": "straight", "road_visibility": 1.0, "speed": 0.0},
    {"road_direction": "right", "road_visibility": 0.61, "speed": 38.0},
    {"road_direction": "unknown", "road_visibility": 0.10, "speed": 22.0},
    {"road_direction": "left", "road_visibility": 0.20, "speed": 62.0},
    {"road_direction": "right", "road_visibility": 0.94, "speed": 13.0},
]
test_items = []
for state in STATES:
    for rawq in Q.values():
        question = normalize_question(rawq)
        ids, marks = build_sequence(tokenizer, state, question, max_len=512, head_max_len=192)
        assert len(marks) == len(question["crit"])
        test_items.append(dict(ids=ids, markers=marks, qtype=QTYPES[question["t"]]))
test_batch = collate_items(test_items, int(tokenizer.pad_token_id))
test_batch = {k: v.to(DEVICE) for k, v in test_batch.items()}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    published_logits, _ = published_model(**test_batch)
    prepared_logits, _ = full_model(**test_batch)
before_diff = float((published_logits.float() - prepared_logits.float()).abs().max())
assert torch.allclose(published_logits.float(), prepared_logits.float(), atol=1e-4, rtol=1e-4), before_diff
reference_autocast_logits = prepared_logits.float().detach().cpu().clone()
print("Shared-weight reference and standalone architecture agree. Max |logit drift|:", before_diff)


In [ ]:
#@title 5. Convert EVERY floating model state tensor to real FP16 and save ONE safetensors checkpoint
# Casting full_model casts the SHARED source encoder/head, so only one large copy is needed.
full_model = full_model.half().eval().requires_grad_(False)
assert all(p.dtype == torch.float16 for p in full_model.parameters() if p.is_floating_point())
assert all(b.dtype == torch.float16 for b in full_model.buffers() if b.is_floating_point())
# Capture production-path FP16 output before serializing and unloading weights.
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    pre_export_logits, _ = full_model(**test_batch)
pre_export_logits = pre_export_logits.float().detach().cpu().clone()
fp32_to_real_fp16_prediction_agreement = float((
    reference_autocast_logits.argmax(-1) == pre_export_logits.argmax(-1)).float().mean())
fp32_to_real_fp16_max_logit_diff = float((reference_autocast_logits - pre_export_logits).abs().max())
print("Original autocast FP32-weights -> real FP16-weight decision agreement:", fp32_to_real_fp16_prediction_agreement)
print("FP16 weight-conversion max logit shift:", fp32_to_real_fp16_max_logit_diff)

full_model.encoder.config.save_pretrained(str(DEST / "encoder"))
tokenizer.save_pretrained(str(DEST / "tokenizer"))
config = {
    "format": "laya-1p2-full-standalone-v1", "architecture": "LayaStandalone",
    "head_context_budget": 64, "head_layers": int(base_config.get("head_layers", 2)),
    "source_head_repo": HEAD_REPO, "source_head_revision": published_info.sha,
    "source_model": SOURCE, "source_model_revision": SOURCE_REVISION,
    "dtype_on_disk": "float16", "inference_parameter_dtype": "torch.float16",
    "model_scope": "option scores only; auxiliary action classifier omitted",
    "serialization": "all floating tensors FP16; integer/bool tensors preserved",
}
(DEST / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
params = sum(p.numel() for p in full_model.parameters())
modules = {k: sum(p.numel() for p in getattr(full_model, k).parameters())
           for k in ("encoder", "head", "type_emb", "scorer")}
state = full_model.state_dict()
buffer_elems = sum(x.numel() for x in full_model.buffers())
state_bytes = sum(t.numel() * t.element_size() for t in state.values())
assert all(v.dtype == torch.float16 for v in state.values() if v.is_floating_point())
weights = DEST / "model.safetensors"
cpu_state = OrderedDict((k, v.detach().to("cpu").contiguous().clone()) for k, v in state.items())
save_file(cpu_state, str(weights), metadata={"format": "pt", "precision": "fp16",
                                            "source_revision": str(SOURCE_REVISION)})
del cpu_state, state
gc.collect()
# Read safetensors dtype metadata without allocating a second full checkpoint.
with safe_open(str(weights), framework="pt", device="cpu") as handle:
    tensor_dtypes = {name: handle.get_slice(name).get_dtype() for name in handle.keys()}
    non_half_float = {name: dtype for name, dtype in tensor_dtypes.items()
                      if dtype in {"F64", "F32", "BF16", "F8_E4M3", "F8_E5M2"}}
    assert not non_half_float, f"Some floating tensors were not saved as FP16: {non_half_float}"
    assert any(dtype == "F16" for dtype in tensor_dtypes.values()), "No FP16 tensors saved"
print("Checkpoint FP16 tensors:", sum(v == "F16" for v in tensor_dtypes.values()))
print("Checkpoint parameters:", f"{params:,}", "; components:", modules)
print("Full FP16 checkpoint bytes:", f"{weights.stat().st_size:,}")


In [ ]:
#@title 6. Delete source model, reload OFFLINE in FP16, and validate decisions and probabilities
# The original and exported wrappers share their underlying weights; free them together.
del full_model, published_model, original, published_mod, pspec
gc.collect()
torch.cuda.empty_cache()
loaded_model, local_tokenizer = standalone_mod.load_model(
    DEST, device=DEVICE, local_files_only=True, compile_head=False)
assert all(p.dtype == torch.float16 for p in loaded_model.parameters() if p.is_floating_point())
assert all(b.dtype == torch.float16 for b in loaded_model.buffers() if b.is_floating_point())
assert local_tokenizer.pad_token_id == tokenizer.pad_token_id
for state in STATES:
    for rawq in Q.values():
        q = normalize_question(rawq)
        a = build_sequence(tokenizer, state, q, max_len=512, head_max_len=192)
        b = standalone_mod.build_sequence(local_tokenizer, state, q, max_len=512, head_max_len=192)
        assert a == b, "Standalone tokenizer/option markers differ from original"
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    local_logits, _ = loaded_model(**test_batch)
local_logits = local_logits.float().detach().cpu()
mask = test_batch["marker_mask"].detach().cpu()
reference_probs = pre_export_logits.masked_fill(~mask, -1e9).softmax(-1)
local_probs = local_logits.masked_fill(~mask, -1e9).softmax(-1)
max_logit_diff = float((pre_export_logits - local_logits).abs().max())
max_probability_diff = float((reference_probs - local_probs).abs().max())
same_predictions = bool(torch.equal(pre_export_logits.argmax(-1), local_logits.argmax(-1)))
# An atol of 0.002 is too strict for independent ModernBERT SDPA FP16 forward passes.
# Weight/dtype roundtrip is checked independently; these are practical output-level guards.
LOGIT_ATOL = 2e-2
LOGIT_RTOL = 2e-2
MAX_PROB_DRIFT = 0.01
close_logits = bool(torch.allclose(pre_export_logits, local_logits,
                                  atol=LOGIT_ATOL, rtol=LOGIT_RTOL))
close_probs = max_probability_diff <= MAX_PROB_DRIFT
print("FP16 pre-save vs FP16 offline reload max logit difference:", max_logit_diff)
print("FP16 pre-save vs FP16 offline reload max probability difference:", max_probability_diff)
print("FP16 decisions identical:", same_predictions, "across", pre_export_logits.shape[0], "examples")
print("FP16 relaxed logit closeness:", close_logits)
# Reject changed decisions or material option-probability drift, not harmless logit rounding.
assert same_predictions and close_probs, (
    "FP16 standalone decision/probability mismatch", max_logit_diff, max_probability_diff, same_predictions)
if not close_logits:
    print("WARNING: FP16 logits exceeded the diagnostic tolerance, but decisions/probabilities passed.")
print("OFFLINE LOCAL-ONLY FP16 RELOAD PASSED")
print("Offline predict example:", loaded_model.predict(local_tokenizer, STATES[0], Q))


In [ ]:
#@title 7. Measure actual FULL MODEL disk size and create machine-readable report
def directory_size(folder):
    return sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file())
def MB(n): return round(n / 1_000_000, 3)
def MiB(n): return round(n / (1024 ** 2), 3)
base_bytes = (SOURCE_DIR / "model.safetensors").stat().st_size
head_bytes = HEAD_FILE.stat().st_size
weight_bytes = weights.stat().st_size
size_record = {
    "model_id": DEST_REPO, "source_head_repo": HEAD_REPO,
    "source_model": SOURCE, "base_revision": SOURCE_REVISION,
    "head_revision": published_info.sha, "original_pinned_notebook_commit": PINNED_COMMIT,
    "total_parameters": int(params), "parameters_by_component": modules,
    "registered_buffer_elements": int(buffer_elems),
    "serialized_state_tensor_bytes": int(state_bytes),
    "model_safetensors_bytes": int(weight_bytes),
    "model_safetensors_MB": MB(weight_bytes),
    "model_safetensors_MiB": MiB(weight_bytes),
    "original_head_file_bytes": int(head_bytes),
    "original_base_file_bytes": int(base_bytes),
    "original_base_plus_head_bytes": int(base_bytes + head_bytes),
    "theoretical_FP32_param_bytes": int(params * 4),
    "theoretical_FP16_param_bytes": int(params * 2),
    "export_dtype": "float16", "floating_state_dtype": "F16",
    "fp16_pre_export_to_reload_max_logit_abs_diff": max_logit_diff,
    "fp16_pre_export_to_reload_max_probability_abs_diff": max_probability_diff,
    "fp16_pre_export_to_reload_predictions_identical": same_predictions,
    "fp16_pre_export_to_reload_logits_close": close_logits,
    "source_autocast_to_real_fp16_prediction_agreement": fp32_to_real_fp16_prediction_agreement,
    "source_autocast_to_real_fp16_max_logit_abs_diff": fp32_to_real_fp16_max_logit_diff,
    "hardware": torch.cuda.get_device_name(0),
    "limitations": "Synthetic choice-only checks; no new held-out accuracy/speed benchmark.",
}
def write_size_report():
    size_record["all_standalone_files_bytes"] = directory_size(DEST)
    size_record["all_standalone_files_MB"] = MB(size_record["all_standalone_files_bytes"])
    (DEST / "size_report.json").write_text(json.dumps(size_record, indent=2), encoding="utf-8")
write_size_report()
print("PARAMETERS:", f"{params:,}")
print("FULL FP16 CHECKPOINT:", f"{weight_bytes:,} bytes / {MB(weight_bytes)} MB / {MiB(weight_bytes)} MiB")
print("STANDALONE DIRECTORY:", f"{directory_size(DEST):,} bytes / {MB(directory_size(DEST))} MB")
print("ORIGINAL FP32-HEAD SOURCE FILE:", f"{head_bytes:,} bytes")
print("ORIGINAL BASE MODEL FILE:", f"{base_bytes:,} bytes")


In [ ]:
#@title 8. Write README, offline instructions, provenance, and dependencies
model_card = f"""---
library_name: pytorch
base_model: {SOURCE}
tags:
- modernbert
- fp16
- typed-decisions
- standalone
- laya-v46
---

# Laya-1.2-speedup — Full Standalone FP16

**Actual FP16 checkpoint**, derived from [{HEAD_REPO}](https://huggingface.co/{HEAD_REPO})
and [{SOURCE}](https://huggingface.co/{SOURCE}). All floating model parameters and
floating buffers are serialized as torch.float16 (half precision), not FP32 weights plus
FP16 autocast. Integer/bool state retains its type.

Includes original 28-layer ModernBERT encoder, optimized 64-token typed decision head,
type embeddings, scorer, tokenizer, encoder configuration, and inference source.
Like the original V4.6 standalone export, it **omits the unused auxiliary action classifier**.

- **Parameter count:** {params:,}
- **Full FP16 checkpoint:** {weight_bytes:,} bytes ({MB(weight_bytes)} MB)
- **Output audit:** 12 synthetic choice examples; FP16 pre-save/reload predictions
  identical = {same_predictions}; max probability difference = {max_probability_diff:.6g}.
- **FP16 conversion caveat:** FP32-weight/autocast vs actual FP16-weight agreement on
  this audit = {fp32_to_real_fp16_prediction_agreement:.3f}. No new general accuracy claim.

## Install and download (once)

```bash
pip install torch 'transformers==4.57.6' 'safetensors>=0.4' 'huggingface_hub>=0.25,<1'
```

```python
from huggingface_hub import snapshot_download
local_dir = snapshot_download('{DEST_REPO}')
```

## Fully offline inference (CUDA GPU recommended)

```python
import sys
sys.path.insert(0, local_dir)
from modeling_laya_standalone import load_model
model, tokenizer = load_model(local_dir, device='cuda', local_files_only=True)
assert next(model.parameters()).dtype == __import__('torch').float16
answers = model.predict(
    tokenizer,
    {{'road_direction': 'left', 'speed': 7.0}},
    {{'steering': {{'type': 'choice', 'instructions': 'Choose road direction.',
                    'criteria': {{'left': 'left', 'straight': 'straight', 'right': 'right'}}}}}})
print(answers)
```

Model folder is completely self-contained for inference: **no TinyCeNN-LM, GitHub,
or original base/head model download** after snapshot_download. GPU FP16 is recommended;
CPU half-precision kernels may be unsupported or slow.

This is an FP16 conversion/export, not a new training run or an additional held-out
benchmark. Small floating-point differences from the FP32-weight version are expected.
Upstream speedup figures are not remeasured here. Check license and redistribution
requirements before making public.

- Source base SHA: `{SOURCE_REVISION}`
- Head SHA: `{published_info.sha}`
- Original export notebook commit: `{PINNED_COMMIT}`
"""
(DEST / "README.md").write_text(model_card, encoding="utf-8")
(DEST / "requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n",
    encoding="utf-8")
(DEST / "SOURCE_REVISIONS.json").write_text(json.dumps({
    "source_base": SOURCE, "base_revision": SOURCE_REVISION,
    "optimized_head": HEAD_REPO, "head_revision": published_info.sha,
    "original_notebook": UPSTREAM_NOTEBOOK, "original_notebook_commit": PINNED_COMMIT,
    "conversion": "all floating checkpoint weights and buffers cast to torch.float16",
}, indent=2), encoding="utf-8")
write_size_report()
print("Standalone FP16 directory ready:", DEST)
print("Standalone total bytes:", directory_size(DEST))


In [ ]:
#@title 9. Optional Hugging Face upload — NEW private FP16 model repo
if PUBLISH_TO_HF:
    notebook_login()
    uploader = HfApi()
    signed_in = uploader.whoami()["name"]
    assert signed_in == "vtava", f"Expected vtava account, got {signed_in}"
    assert DEST_REPO != HEAD_REPO and DEST_REPO != "vtava/Laya-1.2-speedup-standalone", (
        "Refusing to overwrite the original head or original FP32 standalone repo")
    model_url = f"https://huggingface.co/{DEST_REPO}"
    if uploader.repo_exists(DEST_REPO, repo_type="model"):
        existing = set(uploader.list_repo_files(DEST_REPO, repo_type="model"))
        if existing - {".gitattributes", ".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError("FP16 destination repo already contains files. Set "
                               "ALLOW_UPDATE_EXISTING=True only if you intend to update it.")
        print("Resuming existing repo:", model_url)
    else:
        created = uploader.create_repo(
            DEST_REPO, repo_type="model", private=PRIVATE_REPO, exist_ok=False)
        print("Created:", created)
    print("Uploading the complete FP16 checkpoint. This can be a large transfer.")
    uploader.upload_large_folder(repo_id=DEST_REPO, repo_type="model", folder_path=str(DEST))
    expected = {"model.safetensors", "modeling_laya_standalone.py", "config.json",
                "encoder/config.json", "README.md", "size_report.json", "requirements.txt"}
    uploaded = set(uploader.list_repo_files(DEST_REPO, repo_type="model"))
    assert expected.issubset(uploaded), f"Missing uploaded files: {expected - uploaded}"
    print("FP16 standalone uploaded:", model_url)
else:
    print("HF upload disabled. Local standalone model folder:", DEST)
print("Measured full FP16 weight file:", weights.stat().st_size, "bytes")


### Output files

After the notebook succeeds, `/content/Laya_1p2_FP16_Standalone/standalone_model/` contains `model.safetensors` (real FP16 weights), `modeling_laya_standalone.py` (offline local FP16 loader), `config.json`, `encoder/`, `tokenizer/`, `README.md`, `requirements.txt`, `SOURCE_REVISIONS.json`, and `size_report.json`.

If Hugging Face upload is enabled, these files are published to the configured **separate FP16 repository**. A GPU is required for this exporter. No reported parity/speed figure should be treated as a new held-out benchmark without additional testing.
